# The Abinit Speedrun Tutorial


### About the tutorial

In this tutorial, we speedrun through running Abinit workflows and inspecting the tasks along the way.
It is intended for new Abinit users AND expercied users wanting to master automation with Abipy.

**Wiew it on github**
LINK_TO_BE_INSERTED

**Other tutorials and documentation:**

* The official Abinit tutorials.
* The Abipy API documentation.he 
* The abipy_notebook tutorial.

# Session plan

This tutorial is split into several notebooks, meant to be followed in order:

* **0-Setup** (this notebook) -- check your environment.
* **1-Task_to_flow** -- from a single `AbinitTask` to a `Flow`, using silicon.
* **2-Existing_flows** -- flows that take too long to build live, already run for you:
  * ecut and k-point convergence (GaAs)
  * Relaxation -- atomic positions and cell volume (AlN)
  * Band structure -- GaAs, compared with Si from Notebook 1
  * Phonons and Born effective charges from DFPT (GaAs)
* **3-Command_line_interface** -- `abirun.py`, `abiopen.py`, `abicomp.py`, and `Robot`s.
* **4-Challenge** -- put it all together.

Every flow-building function used throughout has a standalone script in
`../Examples/` that you can run from a terminal -- this is the recommended
way to launch anything non-trivial, since it doesn't block a notebook
kernel. `run_*.py` scripts are meant to be run live, during the session;
`make_*.py` scripts build flows that were already run ahead of time (see
`2-Existing_flows`).

## Check your installation

Before generating any input, make sure Abinit is on your `$PATH` and that
`~/.abinit/abipy/manager.yml` / `scheduler.yml` are in place -- see
`../Setup/README.md` and `../Setup/setup_environment.sh` if you haven't run that
yet. Then check your installation with:

```
abicheck.py
```

In [1]:
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for the tutorial

from abipy import abilab
abilab.enable_notebook()  # Tell AbiPy we are running inside a notebook

# Show matplotlib figures embedded in the notebook.
%matplotlib inline

import workshop_lib as wlib

Matplotlib is building the font cache; this may take a moment.


## Load a structure

`workshop_lib` (next to this notebook) gives us `gaas_structure()` and
`si_structure()`, reading from `../Data/Structures/`:

In [9]:
structure = wlib.aln_structure()
print(structure)

Full Formula (Al2 N2)
Reduced Formula: AlN
abc   :   3.128588   3.128588   5.016955
angles:  90.000000  90.000000 120.000000
pbc   :       True       True       True
Sites (4)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Al3+  0.666667  0.333333  0.499287
  1  Al3+  0.333333  0.666667  0.999287
  2  N3-   0.666667  0.333333  0.880713
  3  N3-   0.333333  0.666667  0.380713


In [11]:
#structure.plot()

## Pseudopotentials

The pseudopotentials come from [PseudoDojo](http://www.pseudo-dojo.org)
(norm-conserving, scalar-relativistic, PBE), in `../Data/Pseudos/`:

In [8]:
for p in sorted(wlib.PSEUDO_DIR.glob("*.psp8")):
    print(p.name)

Al.psp8
As.psp8
C.psp8
Ga.psp8
H.psp8
Mg.psp8
N.psp8
O.psp8
Si.psp8


### **Where are the other atoms?**

Download them from Pseudodojo. INSERT_LINK

In this tutorial, we gloss over the details of pseudopotential selection (this is a speedrun afterall).
Hence, we are dealing only with s-p bonded systems. For systems with d and f electrons, several choices must be made: 
* Core/valence partitioning and non-linear core corrections.
* Scalar relativistic (sr) vs. fully relativistic (fr)
* Exchange-correlation functionals
* Norm conserving vs. PAW datasets.